<a href="https://colab.research.google.com/github/Garfieldslard/Crime-Analysis/blob/main/ingestion/notebooks/Ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Loading Data, Filtering & Dropping NA Columns

### 1.1 Load Raw CSVs

In [8]:
import pandas as pd

# Load datasets

crime_df = pd.read_csv('Crime.csv') # You may need to change your file path
dispatch_df = pd.read_csv('Police_Dispatched_Incidents.csv') # You may need to change your file path

/tmp/ipykernel_1843/2184388659.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  crime_df = pd.read_csv('Crime.csv') # You may need to change your file path
/tmp/ipykernel_1843/2184388659.py:6: DtypeWarning: Columns (2,11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  dispatch_df = pd.read_csv('Police_Dispatched_Incidents.csv') # You may need to change your file path


In [9]:
print(crime_df.columns)
print(dispatch_df.columns)

Index(['Incident ID', 'Offence Code', 'CR Number', 'Dispatch Date / Time',
       'Start_Date_Time', 'End_Date_Time', 'NIBRS Code', 'Victims',
       'Crime Name1', 'Crime Name2', 'Crime Name3', 'Police District Name',
       'Block Address', 'City', 'State', 'Zip Code', 'Agency', 'Place',
       'Sector', 'Beat', 'PRA', 'Address Number', 'Street Prefix',
       'Street Name', 'Street Suffix', 'Street Type', 'Latitude', 'Longitude',
       'Police District Number', 'Location'],
      dtype='object')
Index(['Incident_ID', 'Crime Reports', 'Crash Reports', 'Start Time',
       'End Time', 'Priority', 'Initial Type', 'Close Type', 'Address', 'City',
       'State', 'Zip', 'Longitude', 'Latitude', 'Police District Number',
       'Beat', 'PRA', 'CallTime CallRoute', 'Calltime Dispatch',
       'Calltime Arrive', 'Calltime Cleared', 'CallRoute Dispatch',
       'Dispatch Arrive', 'Arrive Cleared', 'Disposition Desc', 'Location'],
      dtype='object')


### 1.2 Convert To Datetime and Filter From 2025 Onward

In [10]:
# Convert to datetime

crime_df['Start_Date_Time'] = pd.to_datetime(crime_df['Start_Date_Time'], errors='coerce')
dispatch_df['Start Time'] = pd.to_datetime(dispatch_df['Start Time'], errors='coerce')

# Filter from 2025 onward

cutoff = pd.Timestamp('2025-01-01')

crime_filtered = crime_df[crime_df['Start_Date_Time'] >= cutoff]
dispatch_filtered = dispatch_df[dispatch_df['Start Time'] >= cutoff]

# Check results

print("Crime filtered rows:", crime_filtered.shape[0])
print("Dispatch filtered rows:", dispatch_filtered.shape[0])

print("Crime date range:", crime_filtered['Start_Date_Time'].min(), "to", crime_filtered['Start_Date_Time'].max())
print("Dispatch date range:", dispatch_filtered['Start Time'].min(), "to", dispatch_filtered['Start Time'].max())

Crime filtered rows: 59811
Dispatch filtered rows: 252157
Crime date range: 2025-01-01 00:00:00 to 2026-04-18 03:26:00
Dispatch date range: 2025-01-01 00:00:52 to 2026-04-20 12:15:07


### 1.3 Check Missing Values Per Column and Total NAs

In [11]:
# Missing values per column

print("\n--- Crime Filtered NA Count ---")
print(crime_filtered.isna().sum())

print("\n--- Dispatch Filtered NA Count ---")
print(dispatch_filtered.isna().sum())

# Total missing values

print("\nTotal NAs in Crime (filtered):", crime_filtered.isna().sum().sum())
print("Total NAs in Dispatch (filtered):", dispatch_filtered.isna().sum().sum())


--- Crime Filtered NA Count ---
Incident ID                   0
Offence Code                  0
CR Number                     0
Dispatch Date / Time       8464
Start_Date_Time               0
End_Date_Time             37945
NIBRS Code                    0
Victims                       0
Crime Name1                   0
Crime Name2                   0
Crime Name3                   0
Police District Name        406
Block Address              3976
City                         22
State                         9
Zip Code                    119
Agency                        0
Place                         0
Sector                        0
Beat                          0
PRA                           0
Address Number             3973
Street Prefix             57591
Street Name                 346
Street Suffix             59023
Street Type                 349
Latitude                      0
Longitude                     0
Police District Number        0
Location                      0
dtype: 

### 1.4 Dropping rows with large amounts of NAs

In [12]:
crime_filtered.columns = crime_filtered.columns.str.lower().str.strip()
dispatch_filtered.columns = dispatch_filtered.columns.str.lower().str.strip()

crime = crime_filtered.drop(columns=[
    'street prefix', 'street suffix', 'end_date_time'
], errors='ignore')

disp = dispatch_filtered.drop(columns=[
    'crime reports', 'crash reports'
], errors='ignore')

---
## 2. Feature Engineering

All new columns are derived — nothing is imputed or fabricated from outside sources.

### 2.1 Dispatch — Datetime Features



In [13]:
disp['start_dt'] = pd.to_datetime(disp['start time'], errors='coerce')
disp['end_dt']   = pd.to_datetime(disp['end time'],   errors='coerce')

disp['hour']        = disp['start_dt'].dt.hour
disp['day_of_week'] = disp['start_dt'].dt.dayofweek          # 0=Mon … 6=Sun
disp['dow_name']    = disp['start_dt'].dt.day_name()
disp['month']       = disp['start_dt'].dt.month
disp['month_name']  = disp['start_dt'].dt.month_name()
disp['is_weekend']  = disp['day_of_week'].isin([5, 6]).astype(int)

# Time-of-day bucket
def hour_bucket(h):
    if   0 <= h <  6: return 'Late Night (0–5)'
    elif 6 <= h < 12: return 'Morning (6–11)'
    elif 12 <= h < 18: return 'Afternoon (12–17)'
    else:              return 'Evening (18–23)'

disp['time_of_day'] = disp['hour'].apply(hour_bucket)

print(disp[['start_dt','hour','dow_name','month_name','is_weekend','time_of_day']].head(5))

             start_dt  hour dow_name month_name  is_weekend        time_of_day
0 2026-04-20 12:15:07    12   Monday      April           0  Afternoon (12–17)
1 2026-04-20 11:13:29    11   Monday      April           0     Morning (6–11)
2 2026-04-20 11:25:33    11   Monday      April           0     Morning (6–11)
3 2026-04-20 11:19:35    11   Monday      April           0     Morning (6–11)
4 2026-04-20 09:08:36     9   Monday      April           0     Morning (6–11)


### 2.2 Dispatch — Response Time & Workload Features

In [14]:
# All time columns are already in SECONDS
# Cap extreme outliers at 99th percentile to reduce skew impact on plots
# (keep originals; capped versions used only for visualization)
RT_CAP = disp['dispatch arrive'].quantile(0.99)

disp['response_time_s']    = disp['dispatch arrive']                  # dispatch → arrived (seconds)
disp['response_time_min']  = disp['dispatch arrive'] / 60             # minutes
disp['response_time_cap']  = disp['response_time_s'].clip(upper=RT_CAP)  # capped for viz

disp['total_incident_s']   = disp['calltime cleared']                 # call → cleared
disp['total_incident_min'] = disp['calltime cleared'] / 60

disp['queue_time_s']       = disp['callroute dispatch']               # routed → dispatched
disp['onscene_time_s']     = disp['arrive cleared']                   # arrived → cleared

print('Response time (dispatch→arrive) stats (minutes):')
print(disp['response_time_min'].describe().round(2))

Response time (dispatch→arrive) stats (minutes):
count    193146.00
mean         11.81
std          27.81
min           0.00
25%           4.08
50%           7.97
75%          14.05
max        4410.63
Name: response_time_min, dtype: float64


A max of 4410 minutes or 73 hours for dipatch response time is not a real response time, it is most likely a data entry error, unclosed call or an incident which was open across multiple different shifts. The standard deviation of 27.81 on a median of 7.97 confirms the distribution is still being pulled hard by these extreme values even after the 99th percentile cap.

In [15]:
# Look at the extreme tail more carefully
rt = disp['dispatch arrive'] / 60  # in minutes

print(rt.quantile([0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 0.999, 1.0]))
print()

# How many rows are above various thresholds?
for threshold in [60, 120, 180, 240, 480]:
    n = (rt > threshold).sum()
    print(f'> {threshold} min ({threshold//60}h): {n:,} rows ({n/len(rt):.2%})')

0.900      24.033333
0.950      34.200000
0.970      43.300000
0.980      51.685000
0.990      68.950000
0.995      90.259167
0.999     157.480667
1.000    4410.633333
Name: dispatch arrive, dtype: float64

> 60 min (1h): 2,695 rows (1.07%)
> 120 min (2h): 415 rows (0.16%)
> 180 min (3h): 134 rows (0.05%)
> 240 min (4h): 50 rows (0.02%)
> 480 min (8h): 20 rows (0.01%)


The distribution is good up to the 99.5th percentile (90 min), then jumps hard, 157 min at 99.9th and 4,410 at the max with only 20 rows above 8 hours. There's no natural cluster between 90 and 4,410, which means those extreme values are probably unclosed records or data entry errors, not real dispatch responses.

In [16]:
RT_THRESHOLD = 120  # minutes

disp['response_time_min'] = disp['dispatch arrive'] / 60
disp['rt_outlier'] = (disp['response_time_min'] > RT_THRESHOLD).astype(int)
disp_clean = disp[disp['rt_outlier'] == 0].copy()

removed = disp['rt_outlier'].sum()
print(f'Flagged as outliers: {removed:,} rows ({removed/len(disp):.2%})')

Flagged as outliers: 415 rows (0.16%)


We removed 415 rows (0.16%) of dispatch while getting rid of values which were impossible, new mean and std will drop substantially and actually reflect the real distribution. You see the new distribution below.

In [17]:
print(f'Analysis dataset: {len(disp_clean):,} rows')
print()
print(disp_clean['response_time_min'].describe().round(2))

Analysis dataset: 251,742 rows

count    192731.00
mean         11.29
std          12.28
min           0.00
25%           4.08
50%           7.95
75%          13.98
max         119.98
Name: response_time_min, dtype: float64


In [18]:
print(f'Total dispatch records:        {len(disp):,}')
print(f'After outlier removal:         {len(disp_clean):,}')
print(f'  With response time recorded: {disp_clean["response_time_min"].notna().sum():,}')
print(f'  No arrival recorded (NaN):   {disp_clean["response_time_min"].isna().sum():,}')

Total dispatch records:        252,157
After outlier removal:         251,742
  With response time recorded: 192,731
  No arrival recorded (NaN):   59,011


### 2.3 Dispatch — Incident Type Features

In [19]:
# Flag whether the incident type changed from initial call to close
disp_clean['type_changed'] = (disp_clean['initial type'] != disp_clean['close type']).astype(int)

# Priority bucket: high / medium / standard / low
priority_map = {0: 'P0-Emergency', 1: 'P1-High', 2: 'P2-Medium', 3: 'P3-Low', 4: 'P4-Standard'}
disp_clean['priority_label'] = disp_clean['priority'].map(priority_map)

# Broad incident category from initial type (first word / known groupings)
def broad_category(t):
    if pd.isna(t): return 'Unknown'
    t = t.upper()
    if 'DOMESTIC' in t:                         return 'Domestic'
    if 'ASSAULT' in t or 'SHOOTING' in t \
       or 'STABBING' in t or 'ROBBERY' in t:    return 'Violent'
    if 'THEFT' in t or 'BURGLARY' in t \
       or 'LARCENY' in t:                       return 'Theft/Burglary'
    if 'TRAFFIC' in t or 'CRASH' in t \
       or 'ACCIDENT' in t:                      return 'Traffic'
    if 'SUSPICIOUS' in t:                       return 'Suspicious'
    if 'WELFARE' in t or 'MENTAL' in t:         return 'Welfare/Mental'
    if 'DISTURBANCE' in t or 'NUISANCE' in t:   return 'Disturbance'
    if 'TRESPASS' in t:                         return 'Trespassing'
    if 'DRUG' in t or 'NARCOTIC' in t:          return 'Drugs'
    if 'PARKING' in t:                          return 'Parking'
    return 'Other'

disp_clean['incident_category'] = disp_clean['initial type'].apply(broad_category)

print('Incident category distribution:')
print(disp_clean['incident_category'].value_counts())

Incident category distribution:
incident_category
Other             79671
Theft/Burglary    41208
Traffic           34775
Welfare/Mental    23102
Suspicious        19072
Domestic          14960
Disturbance       12534
Trespassing       11202
Violent            7917
Parking            7301
Name: count, dtype: int64


### 2.4 Crime — Datetime Features

In [20]:
crime['start_dt']   = pd.to_datetime(crime['start_date_time'], errors='coerce')
crime['hour']       = crime['start_dt'].dt.hour
crime['day_of_week']= crime['start_dt'].dt.dayofweek
crime['dow_name']   = crime['start_dt'].dt.day_name()
crime['month']      = crime['start_dt'].dt.month
crime['is_weekend'] = crime['day_of_week'].isin([5, 6]).astype(int)
crime['time_of_day']= crime['hour'].apply(hour_bucket)

print(crime[['start_dt','hour','dow_name','time_of_day']].head(3))

             start_dt  hour  dow_name       time_of_day
0 2026-04-18 03:26:00     3  Saturday  Late Night (0–5)
1 2026-04-18 01:49:00     1  Saturday  Late Night (0–5)
2 2026-04-17 22:43:00    22    Friday   Evening (18–23)


### 2.5 Geographic Join — Crime Rate per District

We aggregate crime counts per police district and merge onto dispatch, giving each dispatch call a measure of how crime-dense its district is.

In [21]:
crime_by_district = (
    crime.groupby('police district number')
         .size()
         .reset_index(name='district_crime_count')
)

# Crime count per PRA (finer geography)
crime_by_pra = (
    crime.groupby('pra')
         .size()
         .reset_index(name='pra_crime_count')
)
crime_by_pra['pra'] = pd.to_numeric(crime_by_pra['pra'], errors='coerce')

# Merge onto dispatch
disp_clean = disp_clean.merge(crime_by_district, on='police district number', how='left')
disp_clean['pra_float'] = pd.to_numeric(disp_clean['pra'], errors='coerce')
disp_clean = disp_clean.merge(crime_by_pra, left_on='pra_float', right_on='pra', how='left', suffixes=('', '_crime'))

print('District crime count merged onto dispatch:')
print(disp_clean[['police district number','district_crime_count','pra_crime_count']].head(5))

District crime count merged onto dispatch:
  police district number  district_crime_count  pra_crime_count
0                     3D               12451.0            409.0
1                     5D                7594.0             86.0
2                     3D               12451.0            153.0
3                     4D               10154.0             62.0
4                     6D                9339.0            476.0


### 2.6 Create missingness indicators - Dispatch Data

In [22]:
disp_clean['no_arrival']       = disp_clean['calltime arrive'].isna().astype(int)
disp_clean['never_dispatched'] = disp_clean['calltime dispatch'].isna().astype(int)

### 2.7 Creating dispatch_rt - Dispatch Response Time Dataset

In [23]:
disp_rt = disp_clean[
    (disp_clean['calltime arrive'].notna()) &
    (disp_clean['response_time_min'] >= 1)
].copy()

print(f'dispatch_rt: {len(disp_rt):,} rows')

dispatch_rt: 178,030 rows


In [27]:
def hour_bucket(h):
    if   0 <= h <  6: return 'Late Night (0–5)'
    elif 6 <= h < 12: return 'Morning (6–11)'
    elif 12 <= h < 18: return 'Afternoon (12–17)'
    else:              return 'Evening (18–23)'

crime['start_dt']              = pd.to_datetime(crime['start_date_time'], errors='coerce')
crime['hour']                  = crime['start_dt'].dt.hour
crime['day_of_week']           = crime['start_dt'].dt.dayofweek
crime['dow_name']              = crime['start_dt'].dt.day_name()
crime['month']                 = crime['start_dt'].dt.month
crime['is_weekend']            = crime['day_of_week'].isin([5, 6]).astype(int)
crime['time_of_day']           = crime['hour'].apply(hour_bucket)
crime['unlinked_to_dispatch']  = crime['dispatch date / time'].isna().astype(int)

# Verify before saving
print(crime.columns.tolist())
crime.to_csv('crime.csv', index=False)


['incident id', 'offence code', 'cr number', 'dispatch date / time', 'start_date_time', 'nibrs code', 'victims', 'crime name1', 'crime name2', 'crime name3', 'police district name', 'block address', 'city', 'state', 'zip code', 'agency', 'place', 'sector', 'beat', 'pra', 'address number', 'street name', 'street type', 'latitude', 'longitude', 'police district number', 'location', 'start_dt', 'hour', 'day_of_week', 'dow_name', 'month', 'is_weekend', 'time_of_day', 'unlinked_to_dispatch']


### 2.9 Downloading cleaned files

In [28]:
crime_filtered.to_csv('crime_filtered.csv', index=False)
disp_clean.to_csv('dispatch_filtered.csv', index=False)
disp_rt.to_csv('dispatch_rt.csv', index=False)

from google.colab import files

files.download('crime_filtered.csv')
files.download('dispatch_filtered.csv')
files.download('dispatch_rt.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Data Processing Summary — Dispatch Dataset

Raw records:                    252,157
Outlier removal (RT > 120 min):    -415  (0.16%)
Analysis dataset:               251,742

Within analysis dataset:
  Response time recorded:       192,731  (76.6%) → used for response time analysis
  No arrival recorded (NaN):     59,011  (23.4%) → flagged as no_arrival=1

Analytic populations:
  - Temporal/volume analysis:   251,742  (full disp_clean)
  - Response time analysis:     192,731  (disp_clean where no_arrival==0)
  - Unresolved/cancelled calls:  59,011  (disp_clean where no_arrival==1)

## Data Processing Summary — Crime Dataset

Raw records:                     58,380
Records dropped:                      0  (no rows removed)
Analysis dataset:                58,380

Key NA handling:
  - Police District Name (403 NAs) → filled via district number→name mapping
  - Dispatch Date / Time (8,223)   → flagged as unlinked_to_dispatch=1; rows retained
  - End_Date_Time (37,035)         → excluded from any duration analysis; rows retained
  - Street Prefix/Suffix (~57k)    → structural NAs (inapplicable), no action taken
  - City, State, Zip (<120 each)   → minor; retained as-is

Analytic populations:
  - Full crime analysis:            58,380  (all records)
  - Crimes linked to dispatch:      50,157  (unlinked_to_dispatch==0)
  - Crimes not dispatched:           8,223  (unlinked_to_dispatch==1)

---

## Feature Engineering — New Variables Created

### Dispatch Dataset
| Variable | Description | Source |
|---|---|---|
| `response_time_min` | Dispatch → arrival time in minutes | `dispatch arrive / 60` |
| `total_incident_min` | Call received → cleared in minutes | `calltime cleared / 60` |
| `queue_time_s` | Routed → dispatched (seconds) | `callroute dispatch` |
| `onscene_time_s` | Arrived → cleared (seconds) | `arrive cleared` |
| `rt_outlier` | 1 if response time > 120 min | derived |
| `no_arrival` | 1 if arrival time not recorded | derived |
| `never_dispatched` | 1 if call was never dispatched | derived |
| `has_crime_report` | 1 if a crime report number is linked | derived |
| `has_crash_report` | 1 if a crash report number is linked | derived |
| `type_changed` | 1 if initial type ≠ close type | derived |
| `priority_label` | Readable priority (P0-Emergency … P4-Standard) | `priority` |
| `incident_category` | Broad incident grouping (Violent, Domestic, Traffic, etc.) | `initial type` |
| `hour` | Hour of day (0–23) | `start time` |
| `day_of_week` | Day of week (0=Mon, 6=Sun) | `start time` |
| `dow_name` | Day name (Monday … Sunday) | `start time` |
| `month` | Month (1–12) | `start time` |
| `is_weekend` | 1 if Saturday or Sunday | `day_of_week` |
| `time_of_day` | Late Night / Morning / Afternoon / Evening | `hour` |
| `district_crime_count` | Total 2025 crime incidents in same district | joined from crime |
| `pra_crime_count` | Total 2025 crime incidents in same PRA | joined from crime |

### Crime Dataset
| Variable | Description | Source |
|---|---|---|
| `hour` | Hour of day (0–23) | `start_date_time` |
| `day_of_week` | Day of week (0=Mon, 6=Sun) | `start_date_time` |
| `dow_name` | Day name (Monday … Sunday) | `start_date_time` |
| `month` | Month (1–12) | `start_date_time` |
| `is_weekend` | 1 if Saturday or Sunday | `day_of_week` |
| `time_of_day` | Late Night / Morning / Afternoon / Evening | `hour` |
| `unlinked_to_dispatch` | 1 if no dispatch timestamp recorded | `dispatch date / time` |

---

## Output Files

| File | Rows | Description |
|---|---|---|
| `dispatch_clean.csv` | 251,742 | Full dispatch dataset, outliers removed, all features added |
| `dispatch_rt.csv` | 192,731 | Dispatch records with response time recorded (no_arrival==0) |
| `crime.csv` | 58,380 | Full crime dataset, NAs handled, all features added |

---

## Analytic Notes & Limitations

- **Response time analysis** is limited to the 76.6% of dispatch calls where an officer arrival was recorded.
  The 59,011 no-arrival calls are retained separately and may represent cancelled, self-resolved,
  or administratively closed incidents.
- **Outlier threshold of 120 minutes** was selected after inspecting the empirical quantile distribution.
  Values above this threshold (n=415) showed no plausible clustering and likely represent unclosed records.
- **Crime and dispatch datasets are not directly row-matched.** Geographic joins (district, PRA)
  are used to link crime density to dispatch behavior; individual incident linkage is not available
  for all records.
- **Both datasets cover 2025 only**, filtered from the full Data Montgomery open data releases.